# Logistic Regression

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/linear-regression/02-logistic-regression

This notebook implements logistic regression *from scratch* with gradient descent and
mirrors the math derived in the lesson: the sigmoid + log-odds link, binary cross-entropy
from MLE, and the clean gradient collapse `(1/n) X^T (sigma(Xw) - y)`.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Dark matplotlib style to match the site
plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Intuition — regression's classifier cousin

**Logistic regression** turns the straight line of linear regression into a **classifier**. It
computes the same linear score `z = w·x + b`, then squashes it through the **sigmoid** into a
probability in `(0, 1)`. Training minimizes **cross-entropy** (the negative log-likelihood of a
Bernoulli label), and the gradient collapses to the beautifully simple `(p − y)·x` — the same error
signal you've now seen in the neuron, backprop, and MLE notebooks. The decision boundary is a
**straight line** (where `p = 0.5`), which is both its strength (simple, interpretable, calibratable)
and its limit. We build it from scratch and validate against `sklearn`.

## 1. The sigmoid function and the log-odds link

$$\sigma(z) = \frac{1}{1 + e^{-z}}, \qquad \log\frac{\sigma(z)}{1-\sigma(z)} = z$$

The sigmoid squashes any real score into $(0,1)$, and its log-odds (logit) is exactly the
linear score $z$. We verify the log-odds identity numerically below.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-8, 8, 200)
sig = sigmoid(z)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(z, sig, color='#818cf8', linewidth=2.5)
ax.axhline(0.5, color='#2e3347', linestyle='--')
ax.axvline(0, color='#2e3347', linestyle='--')
ax.set_title('Sigmoid Function', color='white')
ax.set_xlabel('z')
ax.set_ylabel('sigma(z)')
plt.tight_layout()
plt.show()

# log-odds(sigma(z)) should equal z
logit = np.log(sig / (1 - sig))
print('max |logit(sigma(z)) - z| = {:.2e}'.format(np.max(np.abs(logit - z))))

**What to notice:** the sigmoid maps any real number to `(0, 1)` — a valid probability — and its
inverse, the **log-odds** `log(p/(1−p))`, is *linear* in the inputs. That's the whole model: a
linear score in log-odds space, squashed into a probability.

## 2. Cross-entropy penalizes confident mistakes

Binary cross-entropy is the negative log-likelihood of a Bernoulli model:

$$\mathcal{L} = -\frac{1}{n}\sum_i \big[ y_i \log \hat y_i + (1-y_i)\log(1-\hat y_i) \big]$$

For a positive example the loss is $-\log(\hat y)$, which explodes as the model becomes
confidently wrong.

In [ ]:
def cross_entropy(y_true, y_pred):
    eps = 1e-15  # avoid log(0)
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

p = np.linspace(1e-3, 1 - 1e-3, 200)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(p, -np.log(p), color='#818cf8', label='y=1: -log(p)')
ax.plot(p, -np.log(1 - p), color='#f43f5e', label='y=0: -log(1-p)')
ax.set_xlabel('predicted P(y=1)')
ax.set_ylabel('loss')
ax.set_ylim(0, 5)
ax.set_title('Cross-entropy loss', color='white')
ax.legend()
plt.tight_layout()
plt.show()

print('loss when predicting 0.01 for a positive: {:.2f}'.format(-np.log(0.01)))

**What to notice:** cross-entropy is gentle when the prediction is confident and correct but
**explodes** when it's confident and wrong (`−log p → ∞` as `p → 0`). That asymmetry is what pushes
the model hardest exactly where it's most mistaken.

## 3. A tiny worked gradient step (matches the lesson by hand)

With

$$X = \begin{bmatrix} 1 & 2 \\ 1 & -1 \end{bmatrix},\; y = [1, 0],\; w = [0,0],\; \eta = 0.5$$

every score is 0, so $\hat y = 0.5$. The gradient is $\tfrac1n X^\top(\hat y - y) = [0, -0.75]$,
giving the update $w \leftarrow [0, 0.375]$. We confirm it.

In [ ]:
X_demo = np.array([[1.0, 2.0],
                   [1.0, -1.0]])
y_demo = np.array([1.0, 0.0])
w_demo = np.zeros(2)
eta = 0.5
n_demo = X_demo.shape[0]

y_hat = sigmoid(X_demo @ w_demo)
grad = (1 / n_demo) * X_demo.T @ (y_hat - y_demo)
w_new = w_demo - eta * grad

print('y_hat       =', y_hat)
print('gradient    =', grad)          # expect [0, -0.75]
print('updated w   =', w_new)         # expect [0, 0.375]

**What to notice:** the single hand-computed gradient step matches the lesson — and the gradient
is just `(p − y)·x` averaged over examples. No sigmoid derivative survives in the final formula; the
sigmoid + cross-entropy pairing makes it cancel, which is why this exact loss is used.

## 4. Logistic regression from scratch via gradient descent

We generate two Gaussian blobs, prepend a bias column, and run the update
$w \leftarrow w - \eta \cdot \tfrac1n X^\top(\sigma(Xw) - y)$ while recording the loss.

In [ ]:
np.random.seed(42)
n = 200
X_pos = np.random.randn(n // 2, 2) + [1.5, 1.0]
X_neg = np.random.randn(n // 2, 2) + [-1.5, -1.0]
X_raw = np.vstack([X_pos, X_neg])
y = np.array([1.0] * (n // 2) + [0.0] * (n // 2))

# design matrix with bias column of ones
X_b = np.c_[np.ones(n), X_raw]
w = np.zeros(X_b.shape[1])
lr = 0.2
epochs = 500
losses = []

for epoch in range(epochs):
    y_pred = sigmoid(X_b @ w)
    grad = (1 / n) * X_b.T @ (y_pred - y)   # the clean collapse
    w -= lr * grad
    losses.append(cross_entropy(y, y_pred))
    if epoch % 100 == 0:
        acc = np.mean((y_pred >= 0.5) == y)
        print('epoch {:3d}: loss={:.4f}  acc={:.3f}'.format(epoch, losses[-1], acc))

final_acc = np.mean((sigmoid(X_b @ w) >= 0.5) == y)
print('final weights [bias, w1, w2] = {}'.format(w.round(3)))
print('final accuracy = {:.3f}'.format(final_acc))

**What to notice:** gradient descent drives the loss down and the accuracy up to ~0.97, and the
final weights `[bias, w1, w2]` point from the negative class toward the positive one. This is the
same GD loop as linear regression — only the loss (cross-entropy) and the sigmoid differ.

## 5. Loss curve

Cross-entropy is convex in $w$, so the loss decreases smoothly to a single minimum.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses, color='#14b8a6', linewidth=2)
ax.set_xlabel('epoch')
ax.set_ylabel('cross-entropy loss')
ax.set_title('Training loss', color='white')
plt.tight_layout()
plt.show()

**What to notice:** the loss curve falls **monotonically** and smoothly — because logistic
regression's loss is **convex** (one global minimum, from the optimization course), gradient descent
can't get stuck. Every run from any start reaches the same optimum.

## 6. Decision boundary

The boundary is the line $w^\top x = 0$. We shade $P(y=1)$ and draw the 0.5 contour.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_pos[:, 0], X_pos[:, 1], c='#818cf8', s=30, alpha=0.7, label='Class 1')
ax.scatter(X_neg[:, 0], X_neg[:, 1], c='#f43f5e', s=30, alpha=0.7, label='Class 0')

x0, x1 = np.meshgrid(np.linspace(-5, 5, 200), np.linspace(-5, 5, 200))
grid = np.c_[np.ones(x0.size), x0.ravel(), x1.ravel()]
probs = sigmoid(grid @ w).reshape(x0.shape)

ax.contourf(x0, x1, probs, levels=20, alpha=0.18, cmap='RdYlBu')
ax.contour(x0, x1, probs, levels=[0.5], colors=['#14b8a6'], linewidths=2)
ax.set_title('Logistic Regression Decision Boundary', color='white')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.legend()
plt.tight_layout()
plt.show()

**What to notice:** the decision boundary (`p = 0.5`) is a **straight line**, with probability
shading from red to blue on either side. Logistic regression can only draw linear boundaries — for
curved ones you need features engineered non-linearly, a kernel, or a neural network.

## The library way — validate against `sklearn`

`sklearn.LogisticRegression` (with `C=1e12` for no regularization) solves the same maximum-likelihood
problem. The cell checks our from-scratch fit reaches the same accuracy and points the same
direction.

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(C=1e12).fit(X_raw, y)          # C huge = ~unregularized MLE
sk_acc = clf.score(X_raw, y)
print(f'our GD accuracy:      {final_acc:.3f}')
print(f'sklearn accuracy:     {sk_acc:.3f}')
print(f'our weights [w1,w2]:  {w[1:].round(3)}   sklearn: {clf.coef_[0].round(3)}')

# same accuracy, and the weight vectors point the same way (cosine ~ 1)
cos = (w[1:] @ clf.coef_[0]) / (np.linalg.norm(w[1:]) * np.linalg.norm(clf.coef_[0]))
assert abs(final_acc - sk_acc) < 0.02 and cos > 0.99, "must match sklearn"
print(f'\nweight-direction cosine = {cos:.4f}  ->  our logistic regression == sklearn ✓')

**What to notice:** our hand-rolled gradient descent and `sklearn` reach the same accuracy and
their weight vectors are nearly **collinear** (cosine ≈ 1). The magnitudes can differ slightly (our
GD hasn't run to the MLE limit), but the decision boundary is the same — same model, same answer.

## Gotchas & tradeoffs

- **Perfect separation → weights diverge.** If a line *perfectly* splits the classes, the MLE pushes
  the weights toward infinity (ever-more-confident) and never converges. Regularization (an L2
  penalty) is the standard fix.
- **The boundary is linear.** Logistic regression can't represent curved boundaries without
  non-linear features / kernels / a network.
- **Probabilities may need calibration.** The outputs are proper probabilities only if the model is
  well-specified; otherwise calibrate (Platt/isotonic).
- **Threshold ≠ 0.5 in general.** For imbalanced classes or asymmetric costs, tune the decision
  threshold rather than defaulting to 0.5.

In [ ]:
# Perfect separation: the weights grow without bound (MLE is at infinity)
Xs = np.array([[-2.0], [-1.0], [1.0], [2.0]]); ys = np.array([0.0, 0.0, 1.0, 1.0])
Xsb = np.c_[np.ones(4), Xs]; ws = np.zeros(2)
for step in (100, 1000, 10000):
    ws = np.zeros(2)
    for _ in range(step):
        ws -= 0.5 * (1/4) * Xsb.T @ (1/(1 + np.exp(-Xsb @ ws)) - ys)
    print(f'{step:>6} steps: weights = {ws.round(2)}  (||w|| = {np.linalg.norm(ws):.1f})')
print('\n-> on perfectly separable data ||w|| just keeps growing; L2 regularization stops it')

**What to notice:** on perfectly separable data the weight norm keeps climbing with more steps —
there's no finite optimum, so the model becomes pathologically overconfident. This is precisely why
production logistic regression almost always includes an L2 penalty (sklearn's default `C`).

## Key takeaways

- Logistic regression passes a linear score through the **sigmoid** to get $P(y=1)\in(0,1)$.
- It is trained with **cross-entropy** loss, the negative log-likelihood of a Bernoulli model.
- The gradient collapses to $\tfrac1n X^\top(\hat y - y)$ — the same form as linear regression.
- The decision boundary is **linear**: the hyperplane $w^\top x = 0$.
- **Softmax** generalizes the sigmoid to multi-class classification.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Binary cross-entropy

The loss that trains logistic regression averages the surprise of each prediction:

$$L = -\frac{1}{n} \sum_i \Big[\, y_i \log p_i + (1 - y_i) \log(1 - p_i) \,\Big]$$

Implement it. The checks pin the landmarks from section 2: perfect predictions cost ~0, coin-flip predictions cost exactly $\log 2$, and a *confident* mistake costs an order of magnitude more than a mild one.

In [ ]:
def bce(y, p):
    """Binary cross-entropy between labels y (0/1) and predicted probabilities p."""
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)

    # TODO(you): -mean of y*log(p) + (1-y)*log(1-p)
    return ...

In [ ]:
# Checks — run me
assert bce([1, 0, 1], [0.999999, 0.000001, 0.999999]) < 1e-5, "near-perfect predictions -> near-zero loss"
assert abs(bce([1, 0], [0.5, 0.5]) - np.log(2)) < 1e-12, "coin-flip predictions cost log 2 ≈ 0.693"

confident_wrong = bce([1.0], [0.01])
mild_wrong = bce([1.0], [0.4])
assert confident_wrong > 4 and mild_wrong < 1, "confident mistakes are punished much harder"
# Edge case: all-same-class labels (degenerate label set, e.g. one rare class in production)
p_all0 = [0.1, 0.2, 0.3]
expected_all0 = -np.mean(np.log(1 - np.array(p_all0)))
assert abs(bce([0, 0, 0], p_all0) - expected_all0) < 1e-12, "all-negative labels must reduce to -mean(log(1-p))"

p_all1 = [0.9, 0.8, 0.7]
expected_all1 = -np.mean(np.log(np.array(p_all1)))
assert abs(bce([1, 1, 1], p_all1) - expected_all1) < 1e-12, "all-positive labels must reduce to -mean(log(p))"

print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def bce(y, p):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
```

</details>

### Exercise 2 — The logistic-regression gradient

Cross-entropy plus sigmoid collapses into the same elegant form as linear regression — *prediction minus target*:

$$\nabla_{\mathbf{w}} L = \frac{1}{n} X^\top (\mathbf{p} - \mathbf{y}), \qquad \mathbf{p} = \sigma(X\mathbf{w})$$

Implement it. The checks compare every component against finite differences of the loss, and confirm a small step against your gradient actually lowers it.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def logistic_gradient(X, y, w):
    """Gradient of mean binary cross-entropy wrt w."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    # TODO(you): predicted probabilities p = sigmoid(X @ w)
    p = ...

    # TODO(you): X^T (p - y) / n
    return ...

In [ ]:
# Checks — run me
rng = np.random.default_rng(1)
Xl = np.column_stack([np.ones(40), rng.standard_normal((40, 2))])
yl = (Xl[:, 1] + Xl[:, 2] > 0).astype(float)
w0 = np.array([0.1, -0.2, 0.3])

def nll(w):
    p = sigmoid(Xl @ w)
    return -np.mean(yl * np.log(p) + (1 - yl) * np.log(1 - p))

g = logistic_gradient(Xl, yl, w0)
h = 1e-6
for i in range(3):
    e = np.zeros(3); e[i] = h
    num = (nll(w0 + e) - nll(w0 - e)) / (2 * h)
    assert abs(g[i] - num) < 1e-6, f"component {i} must match the numerical gradient"

assert nll(w0 - 0.5 * g) < nll(w0), "a small step against the gradient lowers the loss"
# Edge case: perfectly separable data -- weights grow without bound, but the gradient must
# stay finite and keep lowering the loss for many steps (no NaN/blow-up from overflow)
rng_sep = np.random.default_rng(2)
x_sep = rng_sep.standard_normal(40)
y_sep = (x_sep > 0).astype(float)  # perfectly separable by x > 0
X_sep = np.column_stack([np.ones(40), x_sep])
w_sep = np.zeros(2)

def nll_sep(w):
    p = sigmoid(X_sep @ w)
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -np.mean(y_sep * np.log(p) + (1 - y_sep) * np.log(1 - p))

loss_before = nll_sep(w_sep)
for _ in range(200):
    w_sep = w_sep - 1.0 * logistic_gradient(X_sep, y_sep, w_sep)
loss_after = nll_sep(w_sep)
assert np.all(np.isfinite(w_sep)), "gradient must stay finite on perfectly separable data"
assert loss_after < loss_before, "loss must keep decreasing even though it never reaches exactly 0"

print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def logistic_gradient(X, y, w):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    p = sigmoid(X @ w)
    return X.T @ (p - y) / len(y)
```

</details>

---
## 🌐 Extra practice — from Open-Deep-ML

Two more drills from [DML](https://github.com/Open-Deep-ML/DML-OpenProblem) that mirror the sections above but with DML's exact function signatures: a standalone prediction function, and a from-scratch trainer that returns the loss trace alongside the fitted weights.

### Exercise 3 — Binary prediction from weights (DML #104)

Section 6 built a decision boundary from an already-fitted `w`. DML #104 packages just the *prediction* step into its own reusable function: given `X`, `weights`, and `bias` separately (not a single bias-augmented `w`), compute $\sigma(Xw+b)$ and threshold at 0.5. The checks include the all-same-class edge case — every row getting the same score should predict the same label for all of them.

In [ ]:
def predict_logistic(X, weights, bias):
    """DML #104 signature: separate weights/bias, returns 0/1 predictions."""
    X = np.asarray(X, dtype=float)
    weights = np.asarray(weights, dtype=float)

    # TODO(you): z = X @ weights + bias, clip to [-500, 500] to avoid exp() overflow,
    # then threshold sigmoid(z) at 0.5 and cast to int
    return ...

In [ ]:
# Checks — run me
preds = predict_logistic(np.array([[1, 1], [2, 2], [-1, -1], [-2, -2]]), np.array([1, 1]), 0)
assert np.array_equal(preds, [1, 1, 0, 0]), "DML #104 example"

# Edge case: all-same-class labels -- zero weights means every row gets the same score (= bias),
# so every prediction must be identical
X_any = np.array([[5.0, -3.0], [0.2, 10.0], [-7.0, 2.0]])
preds_pos = predict_logistic(X_any, np.array([0.0, 0.0]), bias=2.0)   # sigmoid(2) > 0.5
preds_neg = predict_logistic(X_any, np.array([0.0, 0.0]), bias=-2.0)  # sigmoid(-2) < 0.5
assert np.all(preds_pos == 1), "every row scores sigmoid(2) > 0.5 -> all predicted class 1"
assert np.all(preds_neg == 0), "every row scores sigmoid(-2) < 0.5 -> all predicted class 0"
print("✅ Exercise 3 passed")

<details>
<summary>💡 Show solution</summary>

```python
def predict_logistic(X, weights, bias):
    X = np.asarray(X, dtype=float)
    weights = np.asarray(weights, dtype=float)
    z = np.clip(X @ weights + bias, -500, 500)
    probabilities = 1 / (1 + np.exp(-z))
    return (probabilities >= 0.5).astype(int)
```

</details>

### Exercise 4 — Full training loop with a loss trace (DML #106)

Section 4 already trains logistic regression with gradient descent, but this variant matches DML's *exact* signature and return shape: `train_logreg(X, y, learning_rate, iterations)` internally prepends the bias column itself, uses **unnormalized** gradients ($X^\top(\hat y - y)$, no `1/n`), and returns `(weights, losses)` — both rounded to 4 decimals, with `losses` collected every iteration. The checks pin an exact numeric trace and confirm the loss never increases (a required property of full-batch gradient descent on a convex loss with a small-enough learning rate).

In [ ]:
def train_logreg(X, y, learning_rate, iterations):
    """DML #106 signature: prepends bias internally, unnormalized BCE gradient."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float).reshape(-1, 1)
    Xb = np.hstack([np.ones((X.shape[0], 1)), X])
    B = np.zeros((Xb.shape[1], 1))
    losses = []

    for _ in range(iterations):
        # TODO(you): p = sigmoid(Xb @ B)
        p = ...
        # TODO(you): B -= learning_rate * Xb.T @ (p - y)   (note: NOT divided by n, unlike section 4)
        # TODO(you): append round(bce_sum, 4) to losses, where bce_sum = -sum(y*log(p) + (1-y)*log(1-p))
        ...

    return np.round(B.flatten(), 4).tolist(), losses

In [ ]:
# Checks — run me
X_106 = np.array([[0.7674, -0.2341, -0.2341, 1.5792], [-1.4123, 0.3142, -1.0128, -0.9080],
                  [-0.4657, 0.5425, -0.4694, -0.4634], [-0.5622, -1.9132, 0.2419, -1.7249],
                  [-1.4247, -0.2257, 1.4656, 0.0675], [1.8522, -0.2916, -0.6006, -0.6017],
                  [0.3756, 0.1109, -0.5443, -1.1509], [0.1968, -1.9596, 0.2088, -1.3281],
                  [1.5230, -0.1382, 0.4967, 0.6476], [-1.2208, -1.0577, -0.0134, 0.8225]])
y_106 = np.array([1, 0, 0, 0, 1, 1, 0, 0, 1, 0])
w_106, losses_106 = train_logreg(X_106, y_106, 0.001, 10)
assert np.allclose(w_106, [-0.0097, 0.0286, 0.015, 0.0135, 0.0316]), "DML #106's own tests.json trace"
assert losses_106[:3] == [6.9315, 6.9075, 6.8837], "loss values must match exactly, rounded to 4 decimals"

# Monotonic-loss property: with a small learning rate, full-batch GD on a convex loss never increases the loss
X_small = np.array([[1.0, 0.5], [-0.5, -1.5], [2.0, 1.5], [-2.0, -1.0]])
y_small = np.array([1, 0, 1, 0])
_, losses_small = train_logreg(X_small, y_small, 0.01, 100)
assert all(a >= b - 1e-9 for a, b in zip(losses_small, losses_small[1:])), \
    "loss must be non-increasing at every step"
print("✅ Exercise 4 passed")

<details>
<summary>💡 Show solution</summary>

```python
def train_logreg(X, y, learning_rate, iterations):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float).reshape(-1, 1)
    Xb = np.hstack([np.ones((X.shape[0], 1)), X])
    B = np.zeros((Xb.shape[1], 1))
    losses = []

    for _ in range(iterations):
        p = sigmoid(Xb @ B)
        B -= learning_rate * Xb.T @ (p - y)
        loss = -np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
        losses.append(round(float(loss), 4))

    return np.round(B.flatten(), 4).tolist(), losses
```

</details>